
# 20-Introduction-to-SciPy

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/01-Foundations/20_Introduction_to_SciPy.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=01-Foundations/20_Introduction_to_SciPy.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [ ]:
# --- Global Notebook Setup ---
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate, interpolate, linalg, stats
from scipy.optimize import minimize, root

# Apply the standard course style for all plots
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "font.size": 12,
        "axes.titlesize": 16,
        "axes.labelsize": 12,
        "lines.linewidth": 2,
        "lines.markersize": 6,
    }
)
%config InlineBackend.figure_format = 'retina'  # High-res plots

np.set_printoptions(suppress=True, linewidth=120, precision=4)
warnings.filterwarnings("ignore", category=FutureWarning)


### Table of Contents
1. [The Lens: SciPy as the Scientific Toolkit](#The-Lens:-SciPy-as-the-Scientific-Toolkit)
2. [The SciPy Ecosystem](#The-SciPy-Ecosystem)
3. [Optimization with `scipy.optimize`](#Optimization-with-scipy.optimize)
    - [Unconstrained Minimization](#Unconstrained-Minimization)
    - [Root Finding](#Root-Finding)
    - [Constrained Optimization](#Constrained-Optimization)
    - [Advanced Topic: The Intuition Behind Newton's Method](#Advanced-Topic:-The-Intuition-Behind-Newton's-Method)
4. [Linear Algebra with `scipy.linalg`](#Linear-Algebra-with-scipy.linalg)
5. [Interpolation with `scipy.interpolate`](#Interpolation-with-scipy.interpolate)
6. [Integration with `scipy.integrate`](#Integration-with-scipy.integrate)
7. [Statistics with `scipy.stats`](#Statistics-with-scipy.stats)
8. [Summary](#Summary)
9. [Exercises](#Exercises)


## The Lens: 20-Introduction-to-SciPy
While NumPy provides the foundational array structure for high-performance computing, it is relatively low-level. It gives you the bricks, but **SciPy** (Scientific Python) gives you the tools to build the house. 

SciPy builds on NumPy arrays to provide a vast library of high-level algorithms for optimization, integration, interpolation, eigenvalue problems, algebraic equations, and statistics. For an economist, SciPy is the toolbox that turns mathematical models into computational realities. Whether you are solving for a market equilibrium (root finding), estimating parameters (optimization), or analyzing income distributions (statistics), SciPy likely has a robust, tested implementation ready for you to use.


**Economic question.** In *20-Introduction-to-SciPy*, what must remain economically invariant when the computational representation changes? The economic value of this topic is not the syntax itself but the reliability of the research workflow it enables. Small implementation choices determine whether a result can be reproduced, scaled to a panel or simulation, and audited by another researcher. As you work through the examples, ask which representation makes the economic object easiest to validate and which failure modes would silently change a quantitative conclusion.

### Learning Objectives
* **Solve** optimization problems (minimize, root-finding) using `scipy.optimize`.
* **Perform** numerical integration, interpolation, and ODE solving with SciPy.
* **Apply** statistical distributions and hypothesis tests from `scipy.stats`.

### Prerequisites
* **NumPy:** Array creation, indexing, and linear algebra (Module 01 - NumPy).
* **Calculus:** Derivatives, integrals, and basic differential equations.
* **Learning-path prerequisite:** [`19_Introduction_to_SQL.ipynb`](19_Introduction_to_SQL.ipynb)


> **Learning path:** Building on [`19_Introduction_to_SQL.ipynb`](19_Introduction_to_SQL.ipynb); next continue with [`21_Symbolic_Computation_with_SymPy.ipynb`](21_Symbolic_Computation_with_SymPy.ipynb).


### The SciPy Ecosystem

SciPy is organized into sub-packages covering different scientific domains. The ones most relevant to economics are:

- **`scipy.optimize`**: Optimization and root-finding routines.
- **`scipy.stats`**: Statistical distributions and functions.
- **`scipy.linalg`**: Advanced linear algebra routines.
- **`scipy.integrate`**: Numerical integration and differential equation solvers.
- **`scipy.interpolate`**: Interpolation tools (e.g., splines).

Using these built-in functions is almost always preferable to writing your own. They are highly optimized, widely tested, and handle edge cases that naive implementations often miss.


### Optimization with `scipy.optimize`

Optimization is at the heart of economics: utility maximization, cost minimization, and maximum likelihood estimation all rely on it.


#### Advanced Topic: The Intuition Behind Newton's Method

Many optimization algorithms in SciPy (like `BFGS` used by `minimize`) are based on **Newton's Method**. 

To find the root of a function $f(x) = 0$:
1. Start at a guess $x_0$.
2. Approximate the function with its tangent line at $x_0$.
3. Find where the tangent line crosses zero. This is your new guess $x_1$.
4. Repeat until convergence.

Mathematically, the update rule is: $x_{new} = x_{old} - \frac{f(x_{old})}{f'(x_{old})}$.

For **minimization**, we want to find where the *derivative* is zero ($f'(x) = 0$). So we apply Newton's method to $f'(x)$, using the second derivative (curvature) to guide us: $x_{new} = x_{old} - \frac{f'(x_{old})}{f''(x_{old})}$.

This uses curvature information to take bigger steps when the function is flat and smaller steps when it curves sharply.


In [ ]:
def newton_step(f, f_prime, x_old):
    return x_old - f(x_old) / f_prime(x_old)

# Function: f(x) = x^2 - 4 (Roots at +2, -2)
f = lambda x: x**2 - 4
f_prime = lambda x: 2*x

x = 6 # Initial guess
print(f"Start at x = {x}")
for i in range(5):
    x = newton_step(f, f_prime, x)
    print(f"Iteration {i+1}: x = {x:.4f}, f(x) = {f(x):.4f}")


#### Unconstrained Minimization
The `minimize` function provides a unified interface to many scalar minimization algorithms (e.g., BFGS, Nelder-Mead). You simply pass it the objective function and an initial guess.


In [ ]:
# Define a simple quadratic function: f(x) = (x - 2)^2
fun = lambda x: (x - 2) ** 2

print("Finding the minimum of f(x) = (x - 2)^2 starting at x=0:")
result = minimize(fun, x0=0)
print(f"  Success: {result.success}")
print(f"  Optimal x: {result.x[0]:.4f}")
print(f"  Objective function value: {result.fun:.4f}")


#### Root Finding
Solving for equilibrium often means finding the root of a function (i.e., where $f(x) = 0$). For example, equilibrium price is where `Excess_Demand(p) = Demand(p) - Supply(p) = 0`.


In [ ]:
def excess_demand(p):
    demand = 100 - 2 * p
    supply = 3 * p - 50
    return demand - supply


print("Solving for equilibrium price where excess demand is zero:")
root_res = root(excess_demand, x0=10)
print(f"  Equilibrium Price: {root_res.x[0]:.2f}")
print(f"  Excess Demand at Price: {excess_demand(root_res.x[0]):.2f}")


#### Constrained Optimization

Economic problems are rarely unconstrained. Households maximize utility subject to a budget constraint. Firms minimize cost subject to production targets. `minimize` handles this via the `constraints` and `bounds` arguments.

**Example: Maximize Utility subject to Budget**
Maximize $U(x, y) = x^{0.5} y^{0.5}$ s.t. $10x + 20y \le 100$.


In [ ]:
# 1. Define negative utility (since we minimize)
def neg_utility(vars):
    x, y = vars
    return -(x**0.5 * y**0.5)

# 2. Define constraint: 100 - 10x - 20y >= 0
# Scipy expects constraints in the form g(x) >= 0
cons = ({"type": "ineq", "fun": lambda vars: 100 - 10 * vars[0] - 20 * vars[1]})

# 3. Define bounds: x >= 0, y >= 0
bnds = ((0, None), (0, None))

# 4. Solve
res = minimize(neg_utility, x0=[1, 1], bounds=bnds, constraints=cons)

print("Optimal Bundle:")
print(f"  x = {res.x[0]:.2f}, y = {res.x[1]:.2f}")
print(f"  Utility = {-res.fun:.2f}")


### Linear Algebra with `scipy.linalg`

While NumPy provides basic linear algebra, `scipy.linalg` offers a more extensive set of functions and is generally faster because it is compiled with optimized BLAS/LAPACK support. It includes advanced decompositions (LU, Cholesky, Schur) and solvers.


In [ ]:
A = np.array([[1, 2], [3, 4]])
print("Matrix A:\n", A)
print(f"Determinant: {linalg.det(A):.2f}")

inv_A = linalg.inv(A)
print("Inverse of A:\n", inv_A)

print("\nCheck: A @ inv_A (should be Identity):\n", A @ inv_A)


### Interpolation with `scipy.interpolate`

Interpolation is crucial when you have data at discrete points (e.g., a value function on a grid) and need to evaluate it at points in between. `interp1d` allows you to create a function-like object that performs linear or cubic interpolation.


In [ ]:
x = np.linspace(0, 10, 10)
y = np.sin(x)

# Create an interpolation function
f_interp = interpolate.interp1d(x, y, kind="cubic")

# Evaluate at new points
x_new = np.linspace(0, 10, 100)
y_new = f_interp(x_new)

plt.figure(figsize=(10, 5))
plt.plot(x, y, "o", label="Data Points")
plt.plot(x_new, y_new, "-", label="Cubic Interpolation")
plt.legend()
plt.title("Interpolating a Sine Wave")
plt.show()


### Integration with `scipy.integrate`

Numerical integration (quadrature) is needed for calculating expectations in stochastic models or computing welfare areas. `quad` is the general-purpose workhorse for single integrals.


In [ ]:
# Integrate e^(-x) from 0 to infinity
result, error = integrate.quad(lambda x: np.exp(-x), 0, np.inf)

print(f"Integral of e^(-x) from 0 to inf: {result:.4f}")
print(f"Estimated error: {error:.2e}")


### Statistics with `scipy.stats`

`scipy.stats` contains a vast collection of continuous and discrete probability distributions, as well as a comprehensive suite of statistical tests.


In [ ]:
# Generate some random data from a normal distribution
rng = np.random.default_rng(42)
data = stats.norm.rvs(loc=5, scale=2, size=1000, random_state=rng)

# Perform a one-sample t-test
# H0: Population mean = 5
t_statistic, p_value = stats.ttest_1samp(data, popmean=5)

print(f"Sample Mean: {data.mean():.4f}")
print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value > 0.05:
    print("Result: Fail to reject the null hypothesis (mean is likely 5).")
else:
    print("Result: Reject the null hypothesis.")


### Three-Tier Practice Ladder

**1. Mechanism and assumptions (Conceptual):** Explain the central computational idea in **20-Introduction-to-SciPy** and connect it to one explicit economic object or research workflow.

**2. Reproduce and diagnose (Applied):** Reproduce an example involving The SciPy Ecosystem, Optimization with `scipy.optimize`, then change one input and explain the result before running the code.

**3. Robust extension (Challenge):** Extend the example to a larger or less convenient case and document the correctness and performance checks needed before trusting the result.

> Use the existing exercises above when they target the same skill; this ladder makes the intended progression explicit rather than replacing instructor-authored problems.


# Summary

SciPy transforms Python from a general-purpose language into a scientific powerhouse. We've explored:
- **Optimization:** Finding minima and roots, essential for solving economic models.
- **Linear Algebra:** Advanced matrix operations.
- **Interpolation & Integration:** Tools for working with continuous functions numerically.
- **Statistics:** A comprehensive library for probability and inference.

By mastering SciPy, you gain access to decades of accumulated scientific computing knowledge, packaged in a clean, Pythonic interface.


### Exercises

1.  **Utility Maximization:** A consumer has utility $u(c, l) = \ln(c) + \ln(l)$, subject to $c + wl = wT$, where $w=10$ and $T=24$. Use `scipy.optimize.minimize` to find the optimal consumption $c^*$ and leisure $l^*$. (Hint: You can substitute the constraint into the utility function to make it a single-variable problem, or use the `constraints` argument).

2.  **Curve Fitting:** Generate synthetic data for a demand curve $Q = a - bP + \epsilon$. Use `scipy.optimize.curve_fit` to estimate the parameters $a$ and $b$ from the noisy data.

3.  **Expected Utility:** An agent faces a risk where wealth $W$ is normally distributed with mean 100 and standard deviation 10. Utility is $u(W) = 1 - e^{-0.1W}$ (CARA). Use `scipy.integrate.quad` and `scipy.stats.norm.pdf` to calculate the agent's expected utility: $E[u(W)] = \int u(w) f(w) dw$.


## References & Further Reading

- Python Software Foundation. *Python 3 Documentation*.
- Harris, C. R. et al. (2020). Array programming with NumPy. *Nature*, 585, 357–362.
- McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly.
